# Lensless Computational Imaging — Demo

Reconstruct images from lensless DigiCam measurements end-to-end in a fresh Google Colab:
clone & install → pick data → run `inference.py` → visualize → score with `calculate_metrics.py`.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/daminovkamil/lensless-reconstruction"
REPO_NAME = "lensless-reconstruction"

!git clone {REPO_URL}

%cd {REPO_NAME}
%pip install -q -r requirements.txt


## 2. Dataset

Defaults to the small sample bundled in the repo. To run on your own data, uncomment the
block and paste a Google-Drive (or direct) `.zip` link in the `CustomDirDataset` layout
(`lensless/`, `masks/`, optional `lensed/`).

In [ ]:
from pathlib import Path

DATA_DIR = str((Path.cwd() / "demo_sample").resolve())
RECON_DIR = str((Path.cwd() / "data/saved/reconstructions/test").resolve())
HF_REPO = "daminovkamil/lensless-reconstruction"
HF_CHECKPOINT = "model_best.pth"

# import gdown, zipfile
# gdown.download("PASTE_YOUR_ZIP_LINK", "data.zip", quiet=False, fuzzy=True)
# zipfile.ZipFile("data.zip").extractall("custom_data")
# DATA_DIR = str(next(p.parent for p in Path("custom_data").rglob("lensless")))


## 3. Run inference

Downloads the model from HuggingFace and saves reconstructions to
`data/saved/reconstructions/test/<ImageID>.png`. The checkpoint stores the model architecture.

In [ ]:
!python inference.py \
    inferencer.from_pretrained={HF_REPO} \
    inferencer.pretrained_filename={HF_CHECKPOINT} \
    datasets.test.data_dir={DATA_DIR} \
    dataloader.batch_size=1 \
    dataloader.num_workers=0 \
    dataloader.persistent_workers=false \
    dataloader.prefetch_factor=null \
    'inferencer.device_tensors=[lensless,psf,gt]'

# Any HuggingFace repo containing this project's checkpoint format works:
#   HF_REPO = "<user>/<repo>"
#   HF_CHECKPOINT = "<checkpoint-file>.pth"

## 4. Visualize

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image

data_dir = Path(DATA_DIR)
recon_dir = Path(RECON_DIR)
has_gt = (data_dir / "lensed").is_dir()

img_ids = sorted(p.stem for p in recon_dir.glob("*.png"))[:4]
if not img_ids:
    raise RuntimeError(f"No reconstructions found in {recon_dir}")

cols = 3 if has_gt else 2
titles = ["Lensless", "Reconstruction"] + (["Ground truth"] if has_gt else [])

fig, axes = plt.subplots(
    len(img_ids), cols, figsize=(5 * cols, 4 * len(img_ids)), squeeze=False
)
for r, img_id in enumerate(img_ids):
    row = [
        Image.open(data_dir / "lensless" / f"{img_id}.png").rotate(180),
        Image.open(recon_dir / f"{img_id}.png"),
    ]
    if has_gt:
        row.append(Image.open(data_dir / "lensed" / f"{img_id}.png"))
    for c, (img, title) in enumerate(zip(row, titles)):
        axes[r][c].imshow(np.asarray(img))
        axes[r][c].axis("off")
        if r == 0:
            axes[r][c].set_title(title)

plt.tight_layout()
plt.show()


## 5. Metrics

PSNR, SSIM, MSE, and LPIPS on the ROI (printed only when the dataset has `lensed/`).

In [ ]:
from pathlib import Path

gt_dir = Path(DATA_DIR) / "lensed"
if gt_dir.is_dir():
    !python calculate_metrics.py --gt_dir {gt_dir} --pred_dir {RECON_DIR}
else:
    print("No lensed/ folder; skipping metrics.")
